In [9]:
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GROK_API_KEY")

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2, max_retries=3)

auth.authenticate_user()
import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

sh = gc.create("Fresh Basket Orders")
ws = sh.sheet1
ws.append_row(["name", "item", "quantity_kg", "date", "total"])

print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1a8nDjElx-ZiPNzTHh1UVDMpXu3Ii8hL_hYbUGK1ch5k


In [10]:
from typing import Literal

from langchain.tools import tool  # Used when defining your agent's tools
from pydantic import BaseModel, Field

MENU = {
    "tomatoes": 40,
    "onions": 35,
    "potatoes": 30,
    "carrots": 60,
    "cucumbers": 45,
    "green beans": 80,
}

# Sample hours. Replace these with your shop's actual opening hours.
HOURS = {
    "monday": "9 am to 8 pm",
    "tuesday": "9 am to 8 pm",
    "wednesday": "9 am to 8 pm",
    "thursday": "9 am to 8 pm",
    "friday": "9 am to 8 pm",
    "saturday": "9 am to 8 pm",
    "sunday": "10 am to 6 pm",
}


class Order(BaseModel):
    """A vegetable pickup request taken from a customer's chat message."""

    name: str = Field(min_length=1, description="Customer's first name")
    item: Literal[
        "tomatoes", "onions", "potatoes", "carrots", "cucumbers", "green beans"
    ] = Field(description="Vegetable selected from the shop menu")
    quantity_kg: float = Field(
        gt=0,
        allow_inf_nan=False,
        description="Weight in kilograms. Convert grams to kilograms first.",
    )
    date: str = Field(min_length=1, description="Requested pickup date as the customer said it")

In [11]:
from math import isfinite

from gspread.exceptions import APIError
from langchain.tools import tool
from pydantic import ValidationError


@tool
def get_price(item: str, quantity_kg: float = 1.0) -> str:
    """Estimate the price of one vegetable in rupees.

    Args:
        item: Vegetable name from the menu, such as tomatoes or carrots.
        quantity_kg: Weight in kilograms. Convert grams first, e.g. 500 g is 0.5 kg.
    """
    item = item.strip().lower()
    price_per_kg = MENU.get(item)

    if price_per_kg is None:
        return f"Available menu items: {', '.join(MENU)}."

    if not isfinite(quantity_kg) or quantity_kg <= 0:
        return "Enter a weight greater than zero in kilograms."

    total = price_per_kg * quantity_kg
    return f"Estimated price for {quantity_kg:g} kg of {item}: ₹{total:.2f}."


@tool
def is_open(day: str) -> str:
    """Look up Fresh Basket's opening hours for a weekday, such as Sunday."""
    day = day.strip().lower()
    hours = HOURS.get(day)

    if hours is None:
        return "Enter a weekday name, such as Monday or Sunday."

    return f"Fresh Basket's listed hours on {day.title()} are {hours}."


@tool
def log_order(name: str, item: str, quantity_kg: float, date: str) -> str:
    """Save a vegetable pickup request to Google Sheets.

    Call only after the customer confirms their name, vegetable, weight in
    kilograms, and requested pickup date. The shop confirms the pickup.
    """
    try:
        order = Order(
            name=name.strip(),
            item=item.strip().lower(),
            quantity_kg=quantity_kg,
            date=date.strip(),
        )
    except ValidationError:
        return "Check the customer's name, menu item, weight, and pickup date."

    total = round(MENU[order.item] * order.quantity_kg, 2)

    try:
        ws.append_row([
            order.name,
            order.item,
            order.quantity_kg,
            order.date,
            total,
        ])
    except APIError:
        return "I couldn't verify that the request was saved. Check the sheet before retrying."

    return (
        f"Pickup request saved for {order.name}: "
        f"{order.quantity_kg:g} kg of {order.item}, requested for {order.date}. "
        f"Estimated total: ₹{total:.2f}. The shop will confirm pickup."
    )

In [12]:
from langchain.agents import create_agent

SYSTEM = """You are the friendly order assistant for Fresh Basket Vegetable Shop.

- Use get_price for price estimates and is_open for opening hours.
- Convert grams to kilograms before calling a tool. For example, 500 g is 0.5 kg.
- For a pickup request, collect the customer's name, one vegetable from the menu,
  the weight in kilograms, and the requested pickup date. Ask for missing details.
- Before calling log_order, summarize the request and estimated total.
  Ask the customer to confirm that they want it saved.
- Call log_order only after the customer explicitly confirms the summary.
  Save the request once. If the tool cannot verify the save, tell the customer.
- After a successful save, repeat the order details and estimated total.
  Explain that the shop still needs to confirm stock and pickup.
- Reply politely in English or Telugu, matching the customer.
  Keep replies under 60 words when possible.
"""

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM,
)

history = []


def chat(text: str) -> str:
    global history

    text = text.strip()
    if not text:
        return "Please enter a question or a pickup request."

    try:
        result = agent.invoke({
            "messages": history + [{"role": "user", "content": text}],
        })
    except Exception as exc:
        print("Agent error:", type(exc).__name__)
        return (
            "Sorry, I couldn't complete that request. "
            "If you were saving an order, check the sheet before trying again."
        )

    history = result["messages"]
    return history[-1].text


print(chat("Are you open on Sunday? How much do 2 kg of tomatoes cost?"))
print(chat("I'm Jenny. I'd like to pick up 500 g of carrots on 24 Dec."))
print(chat(""))

Yes, we’re open on Sunday from 10 am to 6 pm.  
2 kg of tomatoes cost ₹80.00.
Sure!  
You’d like to pick up 0.5 kg of carrots on 24 Dec.  
Estimated total: ₹30.00.  
Do you want me to save this order?
Please enter a question or a pickup request.


In [13]:
print(chat("Yes, save that pickup request."))

Your order is saved!  
**Jenny – 0.5 kg carrots, pickup 24 Dec**  
Estimated total: ₹30.00.  
We’ll confirm stock and pickup details shortly. Thanks for choosing Fresh Basket!
